In this notebook we implement pybdshadow for calculating building's shadowing in bologna's city for adressing a Urban Thermal Adaptability index 

In [ ]:
!pip install pybdshadow

In [ ]:
import pandas as pd
import geopandas as gpd
import pybdshadow
import time
import os
from tqdm import tqdm
from pyproj import CRS
import numpy as np

import warnings
warnings.filterwarnings("ignore", message="CRS not set for some of the concatenation inputs")
warnings.simplefilter(action='ignore', category=FutureWarning)


In [ ]:
# user-defined paths
base_dir = '../../../../datos-notebooks/1.global-data-cba/'
date = '2024-01-17 14:08:33' # format 'YYYY-MM-DD HH:MM:SS'

In [ ]:
# derived paths

shadow_folder = os.path.join(base_dir, 'MRT-prediction/shadows/')
os.makedirs(shadow_folder, exist_ok=True)

trees_vectors_path = os.path.join(shadow_folder, 'trees-vectors-clean/')
shadows_folder_path = os.path.join(shadow_folder, 'trees-shadows/')
os.makedirs(shadows_folder_path, exist_ok=True)

In [ ]:
# compute shadows -- 2h aprox
date = pd.to_datetime(date).tz_localize('UTC')

# iterate through all vector files in folder
files = [f for f in os.listdir(trees_vectors_path) if f.endswith(".geojson")]

for idx, fname in enumerate(files, start=1):

    print(f"\n=== Processing file {idx}/{len(files)}: {fname} ===")

    outname = fname.replace(".tif", "_shadow.")
    output_path = os.path.join(shadows_folder_path, outname)

    if os.path.exists(output_path):
        print(f"Skipping {fname} → output already exists: {outname}") # CHECK IF OUTPUT ALREADY EXISTS → SKIP FILE
        continue

    part_file = os.path.join(trees_vectors_path, fname)
    trees = gpd.read_file(part_file)
    trees = trees.to_crs(CRS.from_epsg(4326))  # pybdshadow needs EPSG:4326
    trees = pybdshadow.bd_preprocess(trees)

    # SPLIT
    subparts = np.array_split(trees, 5)
    shadow_part = []

    for k, part in enumerate(subparts, start=1):

        try:
            start = time.time()
            shadow_subpart = pybdshadow.bdshadow_sunlight(
                part, date, roof=False, include_building=True
            )
            print(f"Processing time {fname} subpart{k}: {round(time.time() - start, 2)}s")
            shadow_part.append(shadow_subpart)

        except Exception as e:
            print(f"Error in {fname} subpart{k}: {e}")

        time.sleep(10)

    # MERGE RESULTS
    shadows = gpd.GeoDataFrame(
        pd.concat(shadow_part, ignore_index=True),
        crs=shadow_part[0].crs
    )

    # DISSOLVE / UNION
    union = shadows.union_all()
    shadows = gpd.GeoDataFrame(geometry=[union], crs=shadows.crs).explode(index_parts=False)

    # SAVE OUTPUT
    shadows.to_file(output_path, driver="GeoJSON")
    print(f"Saved: {outname}")


In [ ]:
# collect all shadow files in the folder
shadow_files = [f for f in os.listdir(shadows_folder_path)]

total_shadows = []

# iterate through all shadow files found
for fname in tqdm(shadow_files):
    part_file = os.path.join(shadows_folder_path, fname)
    shadow_part = gpd.read_file(part_file)
    total_shadows.append(shadow_part)


In [ ]:
# merge all
shadows = gpd.GeoDataFrame(
    pd.concat(total_shadows, ignore_index=True),
    crs=total_shadows[0].crs
)

# save output
date_str = date.strftime("%Y-%m-%d_%H%M%S")
output_path = os.path.join(shadow_folder, f"shadows_trees_{date_str}.gpkg")
shadows.to_file(output_path, driver="GPKG")

print(f"Saved merged shadows → {output_path}")